# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [57]:
import os
import re
import json
import csv
import pandas as pd
import load_file as lf
import uml_class as uml

def construire_dictionnaire_hierarchise():

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

        dic_hierarchise['QP'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['QP'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)    

    # Tri des listes dans le dictionnaire
    champs = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs if champ in dic_hierarchise}
    dic_hierarchise['pays'] = ['france', 'france métropolitaine', 'france d\'outre-mer', 'france entière']
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [58]:
# Appel de la fonction pour obtenir le dictionnaire hiérarchisé
dic_hierarchise = recuperer_dictionnaire_hierarchise()
# champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP', 'geopoint']
champs_ranges = ['pays', 'regions', 'departements', 'quartiers', 'communes', 'iris', 'geopoints']
temps_ranges = ['annee', 'trimestre', 'mois', 'semaine', 'date']
hierarchie_champs_spa = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}
hierarchie_champs_tem = {temps: (len(temps_ranges) - i) for i, temps in enumerate(temps_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [59]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return fichiers

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Variables utiles

In [60]:
regexAnnee = r'(19\d{2}|20\d{2})$'
regexMois = r'(0[1-9]|1[0-2])'
regexJour = r'(0[1-9]|[12]\d|3[01])'
regexDate = r'(' + regexAnnee[:-1] + r'[-/]?' + regexMois + r'[-/]?' + regexJour + r')'
regexHeure = r'(([10]\d)|(2[0-3]))[:h]([0-5]\d)([:h]([0-5]\d))?'
regexTrim = r'(19\d{2}|20\d{2})_[a-zA-Z]{1}[1-3]'

listeRegexTemporel = [[regexDate, 'date'], [regexAnnee, 'annee'], [regexTrim, 'trimestre']]

iris_df = pd.read_csv('table_passage_1999_2022.csv', sep=',', encoding='utf-8')
iris_values = iris_df.dropna().values.astype(str).tolist()
listeIris = []
for iris in iris_values:
    for elt in iris:
        if elt not in listeIris and elt != "":
            listeIris.append(elt.lower())

Fonctions utiles

In [61]:
import pprint

def estGeopoint(cell):
    cell = str(cell).strip()
    match = (
        re.match(r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$', cell)
        or re.match(r'[0-9]+\s*([a-zA-Z]+\s*[a-zA-Z]+\s)*[0-9]*', cell)
    )
    if match:
        try:
            lat, lon = float(match.group(1)), float(match.group(2))
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return [True, 'geopoints']
        except Exception:
            pass
    return [False, None]

def estIRIS(cell):
    
    if cell in listeIris:
        return [True, 'iris']
    
    return [False, None]


def estSpatial(cell):
    cell = str(cell).lower()
    infoIRIS = estIRIS(cell)
    if infoIRIS[0]:
        return infoIRIS
    infoGeopoint = estGeopoint(cell)
    if infoGeopoint[0]:
        return infoGeopoint
    for champ, valeurs in dic_hierarchise.items():
        if cell in valeurs:
            return [True, champ]
    return [False, None]

def estTemporel(cell):
    if not isinstance(cell, str):
        cell = str(cell)
    for regex, label in listeRegexTemporel:
        if re.match(regex, cell):
            return [True, label]
    if cell.lower() in ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']:
        return [True, 'mois']
    return [False, None]

def recupererAttributsSpatiauxTemporels(headers, df, score_colonne):
    n_rows = min(10, len(df))
    liste_attributs_spatiaux = {}
    liste_attributs_temporels = {}

    for j, header in enumerate(headers):
        col_values = df[header].astype(str).head(n_rows)
        score_spatial = 0
        score_temporel = 0
        spatial_type = None
        temporel_type = None

        for cell in col_values:
            info_spatial = estSpatial(cell)
            if info_spatial[0]:
                score_spatial += 1
                spatial_type = info_spatial[1]
                if score_spatial*10 > 50 and header not in liste_attributs_spatiaux:
                    liste_attributs_spatiaux[header] = [cell, spatial_type]

            info_temporel = estTemporel(cell)
            if info_temporel[0]:
                score_temporel += 1
                temporel_type = info_temporel[1]
                if score_temporel*10 > 50 and header not in liste_attributs_temporels:
                    liste_attributs_temporels[header] = [cell, temporel_type]

    return liste_attributs_spatiaux, liste_attributs_temporels

def rechercherLowGranEtScope(liste_attributs, hierarchie):
    if not liste_attributs:
        return {'LowGranularite': [None, None], 'Scope': [None, None]}
    
    min_att = list(hierarchie.keys())[0]
    max_att = list(hierarchie.keys())[-1]
    le_plus_bas = [min_att, None]
    le_plus_haut = [max_att, None]

    for champ, valeur in liste_attributs.items():
        if hierarchie[valeur[1]] <= hierarchie[le_plus_bas[0]]:
            le_plus_bas = [valeur[1], champ]
        if hierarchie[valeur[1]] >= hierarchie[le_plus_haut[0]]:
            le_plus_haut = [valeur[1], champ]

    result = {'LowGranularite': le_plus_bas, 'Scope': le_plus_haut}
    return result

def chercherEntete(df, max_lignes=20):
    lignes_testees = 0
    old_df = None
    while lignes_testees < max_lignes:
        headers = df.columns.tolist()
        headers_valides = True
        for h in headers:
            if re.match(r'.*(U|u)nnamed.*', str(h)):
                headers_valides = False
            if ' ' in str(h).strip():
                headers_valides = False
        if headers_valides:
            for header in headers:
                if (estTemporel(header)[0] or estSpatial(header)[0]) and old_df is not None:
                    df = old_df.copy()
                    break
            return df, False
        if len(df) < 1:
            break
        new_headers = df.iloc[0].tolist()
        old_df = df.copy()
        df = df[1:].copy()
        df.columns = [str(h) for h in new_headers]
        lignes_testees += 1
    return df, True

def spatialScopeToDict(scope, dataset):
    if scope[1] is None:
        return {'spatialScopeLevel': None, 'spatialScope': None}
    scope_level = scope[0]
    scope_values = list(dataset[scope[1]].astype(str).unique())
    return {
        'spatialScopeLevel': scope_level, 
        'spatialScope': scope_values
    }

def temporalScopeToDict(scope, dataset):
    if scope[1] is None:
        return {'temporalScopeLevel': None, 'temporalScopeStart': None, 'temporalScopeEnd': None}
    scope_level = scope[0]
    scope_values = sorted(dataset[scope[1]].astype(object).unique())
    return {
        'temporalScopeLevel': scope_level,
        'temporalScopeStart': scope_values[0] if scope_values else None,
        'temporalScopeEnd': scope_values[-1] if scope_values else None
    }

def creerDatasetUML(dataset, nom_fichier, extension, granAndScopeSpat, granAndScopeTemp, liste_attributs_spatiaux, liste_attributs_temporels):
    monSpatialScope = uml.DS_Spatial_Scope(None, None).from_dict(spatialScopeToDict(granAndScopeSpat['Scope'], dataset))
    monTemporalScope = uml.DS_Temporal_Scope(None, None, None).from_dict(temporalScopeToDict(granAndScopeTemp['Scope'], dataset))
    title = nom_fichier
    data_content = [uml.Data_Content(champ, valeur[1], 'Spatial') for champ, valeur in liste_attributs_spatiaux.items()]
    data_content += [uml.Data_Content(champ, valeur[1], 'Temporel') for champ, valeur in liste_attributs_temporels.items()]
    monDataset = uml.Dataset(
        title, None, None, extension, None, None, None, None,
        granAndScopeSpat['LowGranularite'][0], monSpatialScope,
        granAndScopeTemp['LowGranularite'][0], monTemporalScope,
        uml.Theme(None, None), data_content
    )
    monDataset.save_to_json(f'metadatas/{nom_fichier}.json')
    return

def process_dataframe(df, nom_fichier, extension, hierarchie_champs_tem, hierarchie_champs_spa):
    headers = df.columns.tolist()
    score_colonne = {k: 0 for k in range(len(headers))}
    df_sample = df.head(10).copy()
    df_sample = df_sample.astype(str)
    liste_attributs_spatiaux, liste_attributs_temporels = recupererAttributsSpatiauxTemporels(headers, df_sample, score_colonne)
    low_gran_and_scope_spa = rechercherLowGranEtScope(liste_attributs_spatiaux, hierarchie_champs_spa)
    low_gran_and_scope_tem = rechercherLowGranEtScope(liste_attributs_temporels, hierarchie_champs_tem)
    creerDatasetUML(
        df, nom_fichier, extension,
        low_gran_and_scope_spa,
        low_gran_and_scope_tem,
        liste_attributs_spatiaux,
        liste_attributs_temporels
    )
    return


In [62]:
# dataset = 'Opendata/Général/Autonomie/PanoFrance2022_Séniors et Retraités.xlsx'
# fichier = dataset.split('/')[-1]
# nom_fichier, extension = fichier.split('.')

# listeSheets = pd.ExcelFile(dataset)
# nbSheets = len(listeSheets.sheet_names)
# for i in range(nbSheets):
#     try:
#         df = lf.find_type(dataset, i)[0].dropna().astype(str)
#         df, feuilleInvalides = chercherEntete(df)
#         if feuilleInvalides:
#             continue

#         process_dataframe(df, 
#                   f"{nom_fichier}_sheet{i}", 
#                   extension,
#                   hierarchie_champs_tem, 
#                   hierarchie_champs_spa)

#     except Exception as e:
#         print(f"Exception dans la feuille {i}: {e}")


In [ ]:
compteur = 0
for dataset in datasets:
    compteur += 1
    fichier = dataset.split('/')[-1]
    nom_fichier, extension = fichier.split('.')
    print(f"Traitement du fichier {nom_fichier[0:20]} | {compteur}/{len(datasets)}")

    try:
        if extension == 'xlsx':
            listeSheets = pd.ExcelFile(dataset)
            nbSheets = len(listeSheets.sheet_names)
            for i in range(nbSheets):
                print(f"\tTraitement de la feuille {i}/{nbSheets}")
                try:
                    df = lf.find_type(dataset, i)[0].astype(str)
                    df, feuilleInvalides = chercherEntete(df)
                    if feuilleInvalides:
                        print(f"Feuille {i} invalide, passage à la suivante.")
                        continue
                    
                    process_dataframe(df, 
                                      f"{nom_fichier}_sheet{i}", 
                                      extension,
                                      hierarchie_champs_tem, 
                                      hierarchie_champs_spa)
                    
                except Exception as e:
                    print(f"Erreur lors de la lecture de la feuille {i} : {e}")
                    continue
        elif extension == 'csv':
            try:
                df = lf.find_type(dataset)[0].astype(str)
                if len(df.columns) < 5:
                    try:
                        df = pd.read_csv(dataset, sep=';', encoding='utf-8')
                    except:
                        df = pd.read_csv(dataset, sep=';', encoding='latin1')

                process_dataframe(df, 
                                  nom_fichier, 
                                  extension,
                                  hierarchie_champs_tem, 
                                  hierarchie_champs_spa)
                
            except Exception as e:
                print(f"Erreur lors du chargement du fichier {nom_fichier[:20]}: {e}")
                continue
        else:
            print(f"Extension non supportée pour {nom_fichier}")
            continue
    except Exception as e:
        print(f"Erreur générale sur {nom_fichier}: {e}")
        continue

Traitement du fichier niveau-daccessibilit | 1/74
Traitement du fichier access-erp-vt | 2/74
Traitement du fichier etablissement-receva | 3/74
Traitement du fichier accessibilite-des-et | 4/74
Traitement du fichier FD_DEC_2019 | 5/74
Traitement du fichier varmod_DEC_2019 | 6/74
Traitement du fichier varmod_NAIS_2019 | 7/74
Traitement du fichier FD_NAIS_2019 | 8/74
Traitement du fichier FD_MAR_2019 | 9/74
Traitement du fichier varmod_MAR_2019 | 10/74
Traitement du fichier PanoFrance2022_Sénio | 11/74
Traitement du fichier PanoFrance2022 | 12/74
Erreur lors de la lecture de la feuille 1 : DataFrame is empty or contains no valid data.
Erreur lors de la lecture de la feuille 2 : DataFrame is empty or contains no valid data.
Erreur lors de la lecture de la feuille 3 : DataFrame is empty or contains no valid data.
Erreur lors de la lecture de la feuille 4 : DataFrame is empty or contains no valid data.
Erreur lors de la lecture de la feuille 5 : DataFrame is empty or contains no valid data.


KeyboardInterrupt: 